In [ ]:
import numpy as np
import ssqpy
from time import time, sleep
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True)

# Build MPC
ssqpy.setSilentMode()

dt = 0.01
MPC_H = 15

V_WGT = 3e-2
U_WGT = 2e-3
TIP_WGT = 4.2
J1_WGT = 0.0
J2_WGT = 2.0

torque_clip = 0.1

model = ssqpy.model.Model(
    MPC_H,
    dt,
    urdf_path="pendubot.urdf",
    actuated_joints=[0],
    solver_mode=ssqpy.model.SolverMode.InverseDynamics,
)

nq = model.getnq()
nv = model.getnv()
nu = model.getnu()

vel_cost = ssqpy.model.costs.SquaredJointVelocityCost(model, V_WGT)
u_cost = ssqpy.model.costs.SquaredControlCost(model, U_WGT)
tip_cost = ssqpy.model.costs.FrameSquaredTranslationErrorCost(
    model, "tip", np.array((0.0, 0.0, 0.2)), TIP_WGT
)
config_cost = ssqpy.model.costs.SquaredConfigurationErrorCost(
    model, np.array((np.pi, 0.0)), np.array((J1_WGT, J2_WGT))
)

for k in range(MPC_H):
    model.addCost(k, vel_cost)
    model.addCost(k, u_cost)
    model.addCost(k, tip_cost)
    model.addCost(k, config_cost)

model.addCost(MPC_H, vel_cost)
model.addCost(MPC_H, tip_cost)
model.addCost(MPC_H, config_cost)

ssqp_params = ssqpy.solvers.ssqpParams()
ssqp_params.tolerance = 1e-1

admm_params = ssqpy.solvers.admmParams()
admm_params.abs_tolerance = 1e-2
admm_params.rel_tolerance = 1e-2
admm_params.warm_start = True
ssqp_params.admmParams = admm_params


mpc = ssqpy.solvers.MPC(
    model, ssqp_params, sqp_iters=4, qp_iters=100
)

In [ ]:
def wrap_to_reference(theta, reference=np.pi):
    return reference + np.arctan2(
        np.sin(theta - reference), np.cos(theta - reference)
    )


In [ ]:
from cloudpendulumclient.client import Client
import cloudpendulumclient.disturbance as Disturbances
from random import choice

with open("../token.txt", "r") as f:
    token = f.readlines()[0]

CELL_IDS = [201, 203]

Tf = 60.0
N_EXPERIMENTS = 10

# --- Disturbance configuration -------------------------------------------
# Two TimedTorqueDisturbance kicks, at fixed times relative to experiment
# start, regardless of controller state. NOTE: the disturbance is an
# external push applied to the whole pendubot, not a control input through
# the single actuator — so it needs one torque value per joint (nq = 2),
# not one per actuator (nu = 1). Passing a length-1 torque was the bug.
DISTURBANCE_TIMES = [20.0, 40.0]   # seconds after experiment start
DISTURBANCE_DURATION = 0.1       # seconds
DISTURBANCE_TORQUE = [0.05, 0.0]    # Nm, one value per joint (nq = 2)
 
client = Client()
 
# Results collected across all experiments
all_timestamps = []
all_data = []
all_disturbance_windows = []  # list of (start_time, end_time) per experiment, in t0-relative seconds
urls = []
 
for exp_idx in range(N_EXPERIMENTS):
    print(f"=== Running experiment {exp_idx + 1}/{N_EXPERIMENTS} ===")
 
    t0 = time()
    timestamps = []
    data = []
    # Known in advance since these are fixed-time disturbances
    disturbance_windows = [(t, t + DISTURBANCE_DURATION) for t in DISTURBANCE_TIMES]
 
    dist = [
        Disturbances.TimedTorqueDisturbance(
            start_time=t, duration=DISTURBANCE_DURATION, torque=DISTURBANCE_TORQUE
        )
        for t in DISTURBANCE_TIMES
    ]
 
    session_token, livestream_url = client.start_experiment(
        user_token=token,
        experiment_type="Pendubot",
        experiment_time=Tf,
        preparation_time=5.0,
        record=True,
        cell_id=choice(CELL_IDS),
        disturbances=dist,
    )
    print("Received response from server!")
    print("Session token: ", session_token)
    print("Livestream url: ", livestream_url)
 
    current_time = 0.0
 
    start_all = time()
    while (time() - start_all) < Tf:
        start = time()
 
        mq = client.get_position(session_token)
        mv = client.get_velocity(session_token)
        mt = client.get_torque(session_token)
 
        try:
            u = mpc.step(np.hstack((mq, mv)))[1].stage(0)
        except RuntimeError:
            u = np.zeros(nv)
 
        tau = model.inverseDynamics(np.array(mq), np.array(mv), u)
        tau = np.clip(tau, -torque_clip, torque_clip)
 
        try:
            client.set_torque(tau[:1], session_token)
        except RuntimeError as exc:
            print(exc)
            break
 
        elapsed = time() - start
 
        timestamps.append(time() - t0)
        data.append(
            (
                np.array(
                    (wrap_to_reference(mq[0]), wrap_to_reference(mq[1], 0.0))
                ),
                mv,
                mt,
            )
        )
 
    url = client.stop_experiment(session_token)
 
    print("Final state:", mq)
 
    all_timestamps.append(timestamps)
    all_data.append(data)
    all_disturbance_windows.append(disturbance_windows)
    urls.append(url)

In [ ]:
import pickle

save_path = f"data-disturbance/swingup_results_{int(time())}.pkl"
with open(save_path, "wb") as f:
    pickle.dump(
        {
            "all_timestamps": all_timestamps,
            "all_data": all_data,
            "urls": urls,
            "dt": dt,
            "Tf": Tf,
            "N_EXPERIMENTS": N_EXPERIMENTS,
        },
        f,
    )
print(f"Saved raw results to {save_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Plotting (done after all control loops have finished, so forwardKinematics
# calls here don't slow down the real-time control loop above)
# ---------------------------------------------------------------------------

def mask_wrap_jumps(x, threshold=np.pi):
    """Insert NaN wherever consecutive samples jump by more than `threshold`,
    so wrapped angles don't get plotted as a vertical line across the range.
    Works column-wise on an (N, n_joints) array."""
    x = x.astype(float).copy()
    jumps = np.abs(np.diff(x, axis=0)) > threshold
    for col in range(x.shape[1]):
        idx = np.where(jumps[:, col])[0]
        x[idx + 1, col] = np.nan
    return x


def angular_distance(angle, target):
    """Shortest wrapped distance between `angle` and `target`, in [0, pi]."""
    return np.abs(np.mod(angle - target + np.pi, 2 * np.pi) - np.pi)


for exp_idx in range(N_EXPERIMENTS):
    timestamps = all_timestamps[exp_idx]
    data = all_data[exp_idx]
    disturbance_windows = all_disturbance_windows[exp_idx]

    configs = np.array([d[0] for d in data])
    vels = np.array([d[1] for d in data])
    torques = np.array([d[2] for d in data])

    # Tip position computed here, outside the control loop
    tip_positions = np.array(
        [model.forwardKinematics(cfg, "tip")[0][2] for cfg in configs]
    )

    # Wrapped distance-to-target for each joint (q1 -> pi, q2 -> 0)
    dist_q1 = angular_distance(configs[:, 0], np.pi)
    dist_q2 = angular_distance(configs[:, 1], 0.0)

    # Break the plotted line at wrap-around jumps in joint position
    configs_plot = mask_wrap_jumps(configs)

    fig, axes = plt.subplots(5, 1, figsize=(10, 14), sharex=True)
    fig.suptitle(f"Experiment {exp_idx + 1}/{N_EXPERIMENTS}")

    axes[0].axhline(y=0.09, color="r", label="Minimum tip height (0.09)")
    axes[0].plot(timestamps, tip_positions, label="Tip height")
    axes[0].set_ylabel("Tip position")
    axes[0].legend()

    axes[1].plot(timestamps, dist_q1, label="q1 dist. from pi")
    axes[1].plot(timestamps, dist_q2, label="q2 dist. from 0")
    axes[1].set_ylabel("Distance to target (rad)")
    axes[1].legend()

    axes[2].plot(timestamps, configs_plot)
    axes[2].set_ylabel("Joint position")
    axes[2].legend(["q1", "q2"])

    axes[3].plot(timestamps, vels)
    axes[3].set_ylabel("Joint velocity")

    axes[4].plot(timestamps, torques)
    axes[4].set_ylabel("Torque")
    axes[4].set_xlabel("Time (s)")

    # Mark disturbance windows on every subplot
    for w_idx, (d_start, d_end) in enumerate(disturbance_windows):
        for ax in axes:
            ax.axvspan(
                d_start + 6.0,
                d_end + 6.0,
                color="orange",
                alpha=0.25,
                label="Disturbance" if w_idx == 0 else None,
            )
        axes[0].axvline(d_start + 6.0, color="orange", linestyle="--", linewidth=1)

    if disturbance_windows:
        axes[0].legend()

    plt.tight_layout()
    plt.savefig(f"data-disturbance/swingup_{exp_idx + 1}.pdf")

plt.show()

In [ ]:
import urllib.request

for i, url in enumerate(urls):
    urllib.request.urlretrieve(url, f"data-disturbance/swingup_{i + 1}.flv")